# Corrección Cuántica de Errores

**Objetivo:** implementar el código de repetición de 3 qubits y demostrar la corrección de errores X.

El dilema QEC: no podemos copiar estados cuánticos (no-cloning), pero sí podemos codificar 1 qubit lógico en múltiples qubits físicos para detectar y corregir errores sin medir el estado.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Código de repetición: |0_L⟩ = |000⟩, |1_L⟩ = |111⟩
# Codificación del qubit lógico
def encode(qc, qubit_logico=0):
    """Codifica q0 en q0,q1,q2 vía CNOT"""
    qc.cx(qubit_logico, 1)
    qc.cx(qubit_logico, 2)
    return qc

# Verificar: |0⟩ → |000⟩, |1⟩ → |111⟩
for init_state in [0, 1]:
    qc = QuantumCircuit(3)
    if init_state == 1:
        qc.x(0)
    encode(qc, 0)
    sv = Statevector.from_instruction(qc)
    print(f'|{init_state}⟩ → {sv.data.round(2)}')

In [ ]:
# Circuito completo: codificación + error + síndrome + corrección
def repetition_code_circuit(p_error=0.1, error_qubit=None):
    qc = QuantumCircuit(5, 2)  # 3 datos + 2 ancilla síndrome
    
    # Estado inicial: |+⟩ lógico (superposición)
    qc.h(0)
    encode(qc, 0)
    qc.barrier()
    
    # Canal de error: flip bit en qubit específico
    if error_qubit is not None:
        qc.x(error_qubit)
    qc.barrier()
    
    # Medición de síndrome (sin revelar el estado lógico)
    # S1 = Z0Z1: ¿q0 y q1 están de acuerdo?
    qc.cx(0, 3)
    qc.cx(1, 3)
    # S2 = Z1Z2: ¿q1 y q2 están de acuerdo?
    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.measure([3,4], [0,1])
    
    return qc

qc_demo = repetition_code_circuit(error_qubit=1)
print(qc_demo.draw())

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

sim = AerSimulator()

# Síndrome: 00=sin error, 01=error en q2, 10=error en q0, 11=error en q1
decoder = {'00': None, '01': 2, '10': 0, '11': 1}

print('Síndrome por error inyectado:')
for error_q in [None, 0, 1, 2]:
    qc = repetition_code_circuit(error_qubit=error_q)
    qc_t = transpile(qc, sim)
    counts = sim.run(qc_t, shots=100).result().get_counts()
    sindrome = max(counts, key=counts.get)
    correc = decoder[sindrome]
    print(f'  Error en q{error_q}: síndrome={sindrome} → corrección en q{correc}')

## Tasa de Error Lógica vs Física

El código de repetición reduce el error lógico cuando p_física < 0.5 (umbral del código de repetición).

In [ ]:
import matplotlib.pyplot as plt

p_range = np.linspace(0, 0.5, 50)
# Sin corrección: P_L = p
p_sin_corr = p_range
# Con repetición 3 qubits: P_L = 3p²(1-p) + p³ = 3p²-2p³
p_con_corr3 = 3*p_range**2 - 2*p_range**3
# Con repetición 5 qubits: P_L = Σ_(k=3)^5 C(5,k) p^k (1-p)^(5-k)
from scipy.special import comb
p_con_corr5 = sum(comb(5,k)*p_range**k*(1-p_range)**(5-k) for k in range(3,6))

plt.figure(figsize=(7,4))
plt.plot(p_range, p_sin_corr, 'r-', label='Sin QEC')
plt.plot(p_range, p_con_corr3, 'b-', label='Repetición 3 qubits')
plt.plot(p_range, p_con_corr5, 'g-', label='Repetición 5 qubits')
plt.axvline(0.5, color='gray', linestyle='--', label='Umbral')
plt.xlabel('p_física'); plt.ylabel('P_L (error lógico)')
plt.title('Código de repetición'); plt.legend()
plt.tight_layout(); plt.savefig('qec_rep.png', dpi=80); plt.show()

## Ejercicio

1. Implementa el código de repetición para errores Z (phase-flip).
2. ¿Por qué el código de Shor = código de repetición bit-flip + phase-flip?
3. ¿Cuál es el umbral del surface code y por qué es mucho mayor que 0.5%?

In [ ]:
# Resumen: el código Shor codifica 1 qubit en 9 qubits físicos
print('Código de Shor (9 qubits) = 3 grupos × (3 qubits de repetición)')
print('Protege contra:')
print('  - 1 error X en cualquier qubit')
print('  - 1 error Z en cualquier qubit')
print('  - Cualquier combinación de 1 error X+Z')
print('\nSurface code umbral: ~1% (hardware actual: 0.1-0.5%)')

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/09_correccion_errores_codigo_repeticion.ipynb`
- **Surface code:** `Cuadernos/laboratorios/14_topologia_surface_codes.ipynb`
- **Siguiente guía:** `12_algoritmo_shor.ipynb`